# AMI Signal Extraction Pipeline

Runs the (fixed) `ami_signal_matcher.py` across a folder of collected annual reports / SFCRs,
using a manifest-driven approach: the notebook auto-matches company / tier / year / doc type
from filenames, **flags anything it can't confidently resolve instead of guessing**, and lets
you fix the flagged rows before scoring runs.

**Before running:** put `ami_signal_matcher.py` and `ami_signal_dictionary.yaml` in the same
folder as this notebook, and point `CORPUS_DIR` (below) at the folder of downloaded PDFs.

---

**Running this notebook.** Sections 4--7 rebuild the manifest, extract text from
the source PDFs, and score the corpus. The PDFs are not redistributed with this
repository (see `data/README.md`), so those sections will not run from a fresh
clone. Their output, `data/ami_results_wide.csv`, is provided, so notebooks
03--06 run without re-scoring.

All paths are resolved relative to the repository root -- no machine-specific
paths are required.


## 0. Environment check
Confirms you're on the `ml` kernel before anything else runs.

In [ ]:
import sys
print("Python executable:", sys.executable)

required = ["yaml", "pdfplumber", "pandas", "openpyxl"]
missing = []
for pkg in required:
    try:
        __import__(pkg)
        print(f"  OK   {pkg}")
    except ImportError:
        missing.append(pkg)
        print(f"  MISSING  {pkg}")

if missing:
    print("\nMissing packages:", missing)
    print("Run the install cell below, then re-run this cell.")
else:
    print("\nAll dependencies present.")


## 1. Install dependencies (only if the check above showed anything missing)

If this errors with `externally-managed-environment`, your Mac's system Python is blocking
direct pip installs — that's a sign you're not actually on the `ml` conda kernel (check cell 0's
output again). From a terminal with the `ml` environment active: `conda install pandas openpyxl
pyyaml -y && pip install pdfplumber`.


In [ ]:
# %pip install -q pyyaml pdfplumber pandas openpyxl


## 2. Config

In [ ]:
from pathlib import Path

# ---- paths: resolved relative to the repo, not to any one machine ----
def _find_repo_root():
    """Walk up from the working directory until the repo layout is found."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return here

ROOT        = _find_repo_root()
DATA        = ROOT / "data"
DICT_PATH   = ROOT / "dictionary" / "ami_signal_dictionary.yaml"
MATCHER_DIR = ROOT / "src"

# Source PDFs are NOT redistributed with this repository (see data/README.md).
# To re-run this notebook, download the documents listed in
# data/manifest/document_manifest.csv into the folder below.
CORPUS_DIR = DATA / "raw" / "reports"

OUTPUT_DIR = DATA
MANIFEST_PATH     = OUTPUT_DIR / "manifest.csv"
RESULTS_WIDE_PATH = OUTPUT_DIR / "ami_results_wide.csv"
RESULTS_LONG_PATH = OUTPUT_DIR / "ami_results_long.csv"

CORPUS_AVAILABLE = CORPUS_DIR.exists() and any(CORPUS_DIR.rglob("*.pdf"))

print(f"Repo root:  {ROOT}")
print(f"Dictionary: {DICT_PATH}  (exists: {DICT_PATH.exists()})")
print(f"Corpus dir: {CORPUS_DIR}  (available: {CORPUS_AVAILABLE})")
if not CORPUS_AVAILABLE:
    print(
        "\n  NOTE: no source PDFs found. Sections 4-7 below (manifest build, text\n"
        "  extraction, scoring) cannot run without them. The scored output they\n"
        "  produce is already provided at data/ami_results_wide.csv, so the\n"
        "  downstream notebooks (03-06) run without re-scoring the corpus."
    )


## 3. Load the signal dictionary and matcher

In [ ]:
import sys
sys.path.insert(0, str(MATCHER_DIR))

from ami_signal_matcher import load_dictionary, score_document

# The path must be passed explicitly. Calling load_dictionary() with no
# argument falls back to a default location and will silently load the
# wrong dictionary (or none) on any machine but the original.
d = load_dictionary(path=DICT_PATH)

print("Dictionary version:", d["meta"]["version"])
print("Functions:", d["meta"]["functions"])


## 4. Build the manifest

Scans `CORPUS_DIR` for `.pdf` and `.txt` files and tries to auto-resolve **company**, **tier**,
**year**, and **doc type** from the filename. Anything it can't confidently resolve is marked
`UNRESOLVED` — same principle as the job-postings location cleaning: automated first pass,
flagged exceptions reviewed manually, nothing silently guessed.


In [ ]:
import re
import pandas as pd

if not CORPUS_AVAILABLE:
    raise FileNotFoundError(
        f"No PDFs found in {CORPUS_DIR}.\n"
        "Source documents are not redistributed with this repository -- see\n"
        "data/README.md. Skip to notebook 03 to use the provided scored output."
    )

# The locked 29-company sample frame. Update here if the sample changes.

COMPANIES = [
    # Tier A — Large Global / Composite (11)
    ("Allianz", "A"), ("AXA", "A"), ("Generali", "A"), ("Zurich", "A"), ("Aviva", "A"),
    ("Mapfre", "A"), ("Talanx", "A"), ("Munich Re", "A"), ("Swiss Re", "A"),
    ("NN Group", "A"), ("Unipol", "A"),
    # Tier B — Mid-Tier / Specialist (9)
    ("Ageas", "B"), ("Sampo", "B"), ("Hiscox", "B"), ("Admiral", "B"),
    ("Gjensidige", "B"), ("Beazley", "B"), ("Tryg", "B"),
    ("VIG", "B"), ("Helvetia Baloise", "B"),
    # Tier C — Digital-Native / Insurtech (9)
    ("Wakam", "C"), ("Marshmallow", "C"), ("Lemonade", "C"), ("Bolttech", "C"),
    ("Zego", "C"), ("Neodigital", "C"), ("Coface", "C"),
    ("Linea Directa Aseguradora", "C"), ("Alm Brand Group", "C"),
]

def match_company(filename):
    fl = filename.lower()
    for name, tier in sorted(COMPANIES, key=lambda c: -len(c[0])):
        key = name.lower().replace("&", "and")
        if key in fl or name.lower() in fl:
            return name, tier
    return None, None

def match_year(filename):
    years = re.findall(r"20(2[1-9]|3[0-5])", filename)
    return f"20{years[0]}" if years else None

def match_doctype(filename):
    fl = filename.lower()
    if "sfcr" in fl:
        return "SFCR"
    if "annual report" in fl or re.search(r"\bar\b", fl):
        return "Annual Report"
    if "financial statement" in fl or re.search(r"\bfs\b", fl):
        return "Financial Statement"
    return None

def build_manifest(corpus_dir):
    rows = []
    files = list(Path(corpus_dir).rglob("*.pdf")) + list(Path(corpus_dir).rglob("*.txt"))
    for fp in sorted(files):
        company, tier = match_company(fp.name)
        year = match_year(fp.name)
        doctype = match_doctype(fp.name)
        notes = []
        if not company:
            notes.append("company not recognized")
        if not year:
            notes.append("year not found in filename")
        if not doctype:
            notes.append("doc type not found in filename")
        status = "OK" if not notes else "UNRESOLVED"
        rows.append({
            "filepath": str(fp.relative_to(ROOT)), "filename": fp.name,
            "company": company, "tier": tier, "year": year, "doc_type": doctype,
            "status": status, "notes": "; ".join(notes),
        })
    return pd.DataFrame(rows)

df_manifest = build_manifest(CORPUS_DIR)
df_manifest.to_csv(MANIFEST_PATH, index=False)

n_unresolved = (df_manifest["status"] == "UNRESOLVED").sum()
print(f"Found {len(df_manifest)} files. {n_unresolved} UNRESOLVED.")
print(f"Manifest saved to: {MANIFEST_PATH}")
df_manifest


## 5. Fix UNRESOLVED rows

Open `ami_outputs/manifest.csv` (or edit `df_manifest` directly below), fill in the blank
`company` / `tier` / `year` / `doc_type` cells for any `UNRESOLVED` row, set `status` to `OK`,
save, then re-run the cell below to reload it.


In [ ]:
# Re-load the manifest after you've hand-fixed the UNRESOLVED rows in the CSV.
df_manifest = pd.read_csv(MANIFEST_PATH, dtype=str)

still_unresolved = df_manifest[df_manifest["status"] != "OK"]
if len(still_unresolved):
    print(f"{len(still_unresolved)} rows still not OK — these will be skipped in scoring:")
    display(still_unresolved[["filename", "notes"]])
else:
    print("All rows resolved.")

df_ready = df_manifest[df_manifest["status"] == "OK"].copy()
print(f"\n{len(df_ready)} files ready to score.")


## 6. Extract text
PDF text extraction with the same whitespace handling the matcher expects.

In [ ]:
import pdfplumber

def extract_text(filepath):
    filepath = Path(filepath)
    if filepath.suffix.lower() == ".txt":
        return filepath.read_text(encoding="utf-8", errors="ignore")
    text_parts = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text_parts.append(page_text)
    return "\n".join(text_parts)


## 7. Score every document
Runs the fixed `score_document()` across the whole ready-to-score manifest.

In [ ]:
results_wide = []
results_long = []
errors = []

for _, row in df_ready.iterrows():
    try:
        text = extract_text(ROOT / row["filepath"])
        if not text.strip():
            errors.append((row["filename"], "extracted text was empty — likely a scanned/image PDF, needs OCR"))
            continue
        r = score_document(text, d)
    except Exception as e:
        errors.append((row["filename"], str(e)))
        continue

    base = {
        "company": row["company"], "tier": row["tier"], "year": row["year"],
        "doc_type": row["doc_type"], "filename": row["filename"],
        "tokens": r["tokens"], "agentic_share_of_automation": r["agentic_share_of_automation"],
        "raw_agentic": r["raw_totals"]["agentic"], "raw_prior_gen_automation": r["raw_totals"]["prior_gen_automation"],
        "dim_role_evolution_per10k": r["dimension_inputs_per10k"]["role_evolution"],
        "dim_tech_stack_per10k": r["dimension_inputs_per10k"]["tech_stack"],
        "dim_governance_per10k": r["dimension_inputs_per10k"]["governance"],
        "stage_assisted": r["maturity_stage_counts"]["assisted"],
        "stage_augmented": r["maturity_stage_counts"]["augmented"],
        "stage_frontier": r["maturity_stage_counts"]["frontier"],
    }
    for func, val in r["automation_depth_per_function_per10k"].items():
        base[f"automation_depth_{func}_per10k"] = val
        results_long.append({
            "company": row["company"], "tier": row["tier"], "year": row["year"],
            "doc_type": row["doc_type"], "function": func, "automation_depth_per10k": val,
        })
    results_wide.append(base)

df_results = pd.DataFrame(results_wide)
df_long = pd.DataFrame(results_long)

print(f"Scored {len(df_results)} documents. {len(errors)} errors.")
if errors:
    print("\nErrors:")
    for fname, msg in errors:
        print(f"  {fname}: {msg}")

df_results.to_csv(RESULTS_WIDE_PATH, index=False)
df_long.to_csv(RESULTS_LONG_PATH, index=False)
print(f"\nSaved: {RESULTS_WIDE_PATH}")
print(f"Saved: {RESULTS_LONG_PATH}")

df_results


## 8. Quick sanity-check view
Not the final HAR/AMI composite (that needs min-max scaling across the full 20-firm panel — do that separately once collection is more complete). This is just to eyeball whether the numbers look sane before moving on.

In [ ]:
if len(df_results):
    pivot = df_long.pivot_table(
        index=["company", "tier"], columns="function", values="automation_depth_per10k", aggfunc="mean"
    ).round(1)
    display(pivot)

    print("\nAgentic share of automation language, by company/year:")
    display(df_results[["company", "tier", "year", "doc_type", "raw_agentic", "raw_prior_gen_automation", "agentic_share_of_automation"]]
            .sort_values(["tier", "company", "year"]))
else:
    print("No results yet — check the manifest and CORPUS_DIR.")


## Known limitations (carried over from this session's test run — not yet fixed)

- **No lemmatization.** `deploy` / `deploys` / `deployed` / `deploying` are four separate strings to the matcher.
- **`dimension_1_automation_depth.function_specific` phrases bypass the agentic gate.** e.g. "algorithmic pricing" counts at full weight even though "algorithmic" alone is supposed to require a nearby qualifier under the gate's own rule. Worth aligning.
- **No self-reference check.** A firm citing AI as an *external/industry* risk ("AI could introduce new liabilities across the sector") scores identically to a firm claiming its *own* AI deployment. This showed up for real in the Marshmallow SFCR test run.
- **Report-length / sampling sensitivity.** If you only extract part of a long report (e.g. front matter of a 300+ page annual report), you will undercount signal that lives in later operational sections. Worth confirming `extract_text()` above is pulling the *entire* PDF, not truncating.
- **HAR / AMI composite scores are not computed here.** Per the matcher's own note, min-max scaling has to happen *after* every firm in the panel is scored, not per-document — do that as a separate step once more of the 20 are collected.
